In [7]:
# ==============================================================
# 📚 RAG LIBRARY MANAGEMENT ASSISTANT
# Gradio + Chroma + Embeddings + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------
# !pip install gradio chromadb sentence-transformers pypdf transformers

# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline

# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
print("Loading embedding model...")
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded")

# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------
client = chromadb.Client()

try:
    client.delete_collection("library_docs")
except:
    pass

collection = client.create_collection("library_docs")

# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------
print("Loading LLM...")
llm = pipeline("text-generation", model="google/flan-t5-base")
print("LLM loaded successfully")

# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF (Library Dataset)
# --------------------------------------------------------------
def process_pdf(file):
    print("Processing PDF...")
    reader = PdfReader(file.name)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    chunks = chunk_text(text)
    print("Total chunks:", len(chunks))
    for i, chunk in enumerate(chunks):
        embedding = embedding_model.encode(chunk).tolist()
        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )
    return f"Library dataset processed successfully. {len(chunks)} chunks stored."

# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------
def retrieve(query, k=3):
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    docs = results["documents"][0]
    print("\nRetrieved Chunks:\n", docs)
    return docs

# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------
def answer_question(query):
    docs = retrieve(query)
    context = " ".join(docs)
    prompt = f"""
You are a Library Management Assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short, clear, and practical answer.
"""
    response = llm(prompt, max_length=200, temperature=0.2)
    result = response[0]["generated_text"]
    print("\nRaw Model Output:\n", result)
    return result

# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------
def chat(question):
    if not question.strip():
        return "Please enter a question."
    answer = answer_question(question)
    if not answer:
        return "No answer generated."
    return answer

# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------
with gr.Blocks() as demo:
    gr.Markdown("# 📚 Library Management RAG Assistant")

    gr.Markdown("""
Upload a library dataset (PDF) and ask questions about:

• most borrowed books
• member borrowing patterns
• late return predictions
• genre popularity
• recommendations
""")

    pdf_input = gr.File(label="Upload Library Dataset PDF")
    upload_button = gr.Button("Process Dataset")
    status = gr.Textbox(label="Status")

    upload_button.click(process_pdf, inputs=pdf_input, outputs=status)

    question_box = gr.Textbox(label="Ask a Library Question")
    answer_box = gr.Textbox(label="Answer", lines=15)
    ask_button = gr.Button("Ask")

    ask_button.click(chat, inputs=question_box, outputs=answer_box)

# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------
demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://723720cda21edb3757.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
